In [47]:
"""
=============================================================================
CCA Panel Builder
=============================================================================
Builds a country × week panel dataset for sovereign Contingent Claims Analysis.

Data sources (all in data/processed/CCA/):
  - Weekly_FX_Rates.csv              → FX rates (defines weekly frequency)
  - IMF_monetary_base_in_domestic_curr.csv → Monetary base
  - IMF_foreign_risk_free_rates.csv  → Risk-free rate candidates
  - IMF_debt_local_and_ext.csv       → Sovereign debt (gross/net/external)
  - QEDS_SDDS.CSV                    → External debt by maturity (short/long-term)

Output: cca_panel.csv  (double-indexed: country × date, weekly frequency)
=============================================================================
"""

'\n=============================================================================\nCCA Panel Builder\n=============================================================================\nBuilds a country × week panel dataset for sovereign Contingent Claims Analysis.\n\nData sources (all in data/processed/CCA/):\n  - Weekly_FX_Rates.csv              → FX rates (defines weekly frequency)\n  - IMF_monetary_base_in_domestic_curr.csv → Monetary base\n  - IMF_foreign_risk_free_rates.csv  → Risk-free rate candidates\n  - IMF_debt_local_and_ext.csv       → Sovereign debt (gross/net/external)\n  - QEDS_SDDS.CSV                    → External debt by maturity (short/long-term)\n\nOutput: cca_panel.csv  (double-indexed: country × date, weekly frequency)\n=============================================================================\n'

# 0. Imports & Configuration

In [48]:
import pandas as pd
import numpy as np
import warnings
import os

warnings.filterwarnings("ignore")

DATA_DIR = "../data/processed/CCA"
OUTPUT_DIR = "../data/processed/CCA"
OUTPUT_FILE = os.path.join(OUTPUT_DIR, "cca_panel.csv")

# 1. Target Countries & Name Mapping

We maintain a canonical list of target countries and mapping dictionaries
to reconcile naming differences across data sources.

In [49]:
# ── Canonical target country list (using FX-file naming convention) ──────────
TARGET_COUNTRIES = [
    "United States", "United Kingdom", "Japan", "Australia", "China",
    "France", "Germany", "India", "Italy", "Turkey",
    "Brazil", "Spain", "Hong Kong", "South Africa", "Mexico",
    "Indonesia", "Argentina", "Poland", "Switzerland", "South Korea",
    "Egypt", "Colombia", "Thailand", "Malaysia", "Chile",
    "Vietnam", "Czech Republic", "Ireland", "Saudi Arabia", "Hungary",
    "Netherlands", "Philippines", "Romania", "Austria", "Greece",
    "Belgium", "Qatar", "Finland", "Peru", "Israel",
    "Bulgaria", "Bahrain", "Lithuania", "Panama", "Morocco",
    "Slovenia", "Kuwait", "Croatia", "Slovakia", "Latvia",
    "Estonia", "Kazakhstan", "Serbia", "Sri Lanka", "Uruguay",
    "Costa Rica", "Dominican Republic", "Pakistan", "El Salvador", "Iraq",
    "Jamaica", "Trinidad and Tobago", "New Zealand", "Sweden", "Norway",
    "Denmark", "Cyprus", "Iceland", "United Arab Emirates", "Taiwan",
    "Abu Dhabi", "Dubai",
]

# Note: User-provided list has "Czechia" — we normalise to "Czech Republic"
#       which is what the FX file uses.

# ── IMF country name → FX country name mapping ─────────────────────────────
# Only entries where the names actually differ need to be listed.
IMF_TO_FX_NAME = {
    "Russian Federation":                       "Russia",
    "Korea, Republic of":                       "South Korea",
    "Türkiye, Republic of":                     "Turkey",
    "Egypt, Arab Republic of":                  "Egypt",
    "Hong Kong Special Administrative Region, People's Republic of China": "Hong Kong",
    "Poland, Republic of":                      "Poland",
    "Serbia, Republic of":                      "Serbia",
    "Kazakhstan, Republic of":                  "Kazakhstan",
    "Czech Republic":                           "Czech Republic",   # same
    "United Arab Emirates":                     "United Arab Emirates",
    # IMF may list UAE or individual emirates; FX has Abu Dhabi/Dubai separately
    # — we keep UAE → United Arab Emirates and handle Abu Dhabi/Dubai via FX only
    "Sri Lanka":                                "Sri Lanka",        # same
    "Iceland":                                  "Iceland",          # same
}

# Reverse mapping (FX → IMF) for lookups when needed
FX_TO_IMF_NAME = {v: k for k, v in IMF_TO_FX_NAME.items()}

# Additional aliases the user's target list might use
ALIAS_TO_FX = {
    "Czechia": "Czech Republic",
}


def standardize_imf_country(name: str) -> str:
    """Map an IMF country name to the FX-file convention."""
    return IMF_TO_FX_NAME.get(name, name)

# 2. Load FX Rates and CDS (Weekly — defines the panel date index)

Format: Date column (YYYY-MM-DD), one column per country.
Units:  domestic currency per 1 USD.

In [ ]:
def load_fx_rates(path: str) -> pd.DataFrame:
    """
    Load weekly FX rates and reshape to long format.

    Returns
    -------
    DataFrame with columns: [date, country, fx_rate_per_usd]
    """
    df = pd.read_csv(path, parse_dates=["Date"])
    df = df.rename(columns={"Date": "date"})

    # Melt: one row per (date, country)
    id_vars = ["date"]
    value_vars = [c for c in df.columns if c != "date"]
    df_long = df.melt(id_vars=id_vars, var_name="country", value_name="fx_rate_per_usd")

    # Apply alias normalisation (e.g. Czechia → Czech Republic)
    df_long["country"] = df_long["country"].replace(ALIAS_TO_FX)

    # Keep only target countries
    df_long = df_long[df_long["country"].isin(TARGET_COUNTRIES)].copy()
    df_long["fx_rate_per_usd"] = pd.to_numeric(df_long["fx_rate_per_usd"], errors="coerce")
    df_long = df_long.sort_values(["country", "date"]).reset_index(drop=True)

    return df_long


def load_cds_spreads(path: str) -> pd.DataFrame:
    """
    Load weekly CDS spreads and reshape to long format.

    Returns
    -------
    DataFrame with columns: [date, country, cds_spread_bps]
    """
    df = pd.read_csv(path, parse_dates=["Date"])
    df = df.rename(columns={"Date": "date"})

    value_vars = [c for c in df.columns if c != "date"]
    df_long = df.melt(id_vars=["date"], var_name="country", value_name="cds_spread_bps")

    # Normalise country names (e.g. Czechia → Czech Republic)
    df_long["country"] = df_long["country"].replace(ALIAS_TO_FX)

    df_long = df_long[df_long["country"].isin(TARGET_COUNTRIES)].copy()
    df_long["cds_spread_bps"] = pd.to_numeric(df_long["cds_spread_bps"], errors="coerce")
    df_long = df_long.dropna(subset=["cds_spread_bps"])
    df_long = df_long.sort_values(["country", "date"]).reset_index(drop=True)

    return df_long

# 3. Helper: Melt IMF Wide-Format CSVs

IMF files store time series per row with columns like "2000-M01", "2020-Q1",
or "2020" for monthly, quarterly, and annual frequencies respectively.

In [51]:
def melt_imf_wide(
    path: str,
    indicator_filter: str | None = None,
    value_col_name: str = "value",
    country_col: str = "COUNTRY",
    indicator_col: str = "INDICATOR",
    scale_col: str = "SCALE",
) -> pd.DataFrame:
    """
    Read an IMF wide-format CSV and melt to long format.

    Parameters
    ----------
    path : str
        Path to CSV.
    indicator_filter : str or None
        If given, keep only rows whose INDICATOR column contains this substring.
    value_col_name : str
        Name for the melted value column.

    Returns
    -------
    DataFrame with columns: [country, indicator, date, <value_col_name>, scale]
    """
    df = pd.read_csv(path, dtype=str)

    # Identify time columns (start with a digit, e.g. "2000-M01", "2000")
    meta_cols = [c for c in df.columns if not c[0].isdigit()]
    time_cols = [c for c in df.columns if c[0].isdigit()]

    if indicator_filter:
        mask = df[indicator_col].str.contains(indicator_filter, case=False, na=False)
        df = df[mask]

    df_long = df.melt(
        id_vars=meta_cols,
        value_vars=time_cols,
        var_name="period",
        value_name=value_col_name,
    )

    # Parse the period column to a proper date
    df_long["date"] = df_long["period"].apply(_parse_imf_period)
    df_long = df_long.dropna(subset=["date"])

    # Clean value
    df_long[value_col_name] = pd.to_numeric(df_long[value_col_name], errors="coerce")

    # Standardise country name
    df_long["country"] = df_long[country_col].apply(standardize_imf_country)

    # Keep useful columns
    keep = ["country", "date", value_col_name]
    if indicator_col in df_long.columns:
        df_long = df_long.rename(columns={indicator_col: "indicator"})
        keep.insert(2, "indicator")
    if scale_col in df_long.columns:
        keep.append(scale_col)

    df_long = df_long[keep].copy()
    df_long = df_long.sort_values(["country", "date"]).reset_index(drop=True)
    return df_long


def _parse_imf_period(p: str) -> pd.Timestamp | None:
    """Convert IMF period strings to Timestamp (end-of-period)."""
    try:
        if "-M" in p:
            # Monthly: "2020-M01" → last day of Jan 2020
            return pd.Timestamp(p.replace("-M", "-")) + pd.offsets.MonthEnd(0)
        elif "-Q" in p:
            # Quarterly: "2020-Q1" → last day of quarter
            year, q = p.split("-Q")
            month = int(q) * 3
            return pd.Timestamp(f"{year}-{month:02d}-01") + pd.offsets.MonthEnd(0)
        else:
            # Annual: "2020" → Dec 31
            return pd.Timestamp(f"{p}-12-31")
    except Exception:
        return None

# 4. Load IMF Monetary Base

### Variable candidates (all in Millions of domestic currency):
- **[CHOSEN] Liabilities, Monetary base (CBS)**
  → Proxy for domestic-currency sovereign liabilities (short-term claim
    on the central bank that the government implicitly guarantees).

No other candidates in this file.

In [52]:
def load_monetary_base(path: str) -> pd.DataFrame:
    """
    Load monetary base data.

    Candidate variables
    -------------------
    [CHOSEN] "Liabilities, Monetary base (CBS)" — Millions of domestic currency
        Rationale: Represents the domestic-currency monetary liabilities of
        the central bank, used as a proxy for short-term domestic sovereign
        obligations in the CCA framework (Gray et al. 2007).

    Returns
    -------
    DataFrame: [country, date, monetary_base_lcu_mn]
    """
    df = melt_imf_wide(
        path,
        indicator_filter="Monetary base",
        value_col_name="monetary_base_lcu_mn",
    )
    # Drop indicator column (only one candidate)
    if "indicator" in df.columns:
        df = df.drop(columns=["indicator"])
    if "SCALE" in df.columns:
        df = df.drop(columns=["SCALE"])

    # Keep only target countries
    df = df[df["country"].isin(TARGET_COUNTRIES)].copy()
    return df

# 5. Load IMF Risk-Free Rates

### Variable candidates (all in Percent per annum):

1. **Monetary policy-related, Rate** — central bank policy rate
2. **Government securities: Treasury bills yields, 3 Months Rate** — short-term govt yield
3. **Government bonds yields, Short to medium term Rate** — medium-term govt yield
4. **Money market Rate** — interbank/money-market rate

### Selection rationale
For CCA we need the **domestic risk-free rate** to discount sovereign
liabilities.  The preferred hierarchy is:
  (a) T-bill 3M yield (most directly comparable to risk-free),
  (b) Policy rate (if T-bill unavailable),
  (c) Money market rate (fallback).

We load ALL candidates into the panel and create a composite column that
picks the best available for each country-date.

In [53]:
# ── Substrings used to identify each candidate in the INDICATOR column ──────
RATE_CANDIDATES = {
    # candidate_label           : indicator_substring
    "policy_rate_pct":           "Monetary policy-related",
    "tbill_3m_pct":              "Treasury bills yields, 3 Months",
    "govt_bond_short_pct":       "Government bonds yields, Short to medium term",
    "money_market_rate_pct":     "Money market",
}

# ── Chosen composite: priority order (first non-NaN wins) ───────────────────
RATE_PRIORITY = [
    "tbill_3m_pct",           # [PREFERRED]  closest to true risk-free
    "policy_rate_pct",        # [FALLBACK 1] central bank policy rate
    "money_market_rate_pct",  # [FALLBACK 2] interbank rate
    "govt_bond_short_pct",    # [FALLBACK 3] short-to-medium govt bond yield
]


def load_risk_free_rates(path: str) -> pd.DataFrame:
    """
    Load all risk-free rate candidates and build a composite column.

    Returns
    -------
    DataFrame: [country, date, policy_rate_pct, tbill_3m_pct,
                govt_bond_short_pct, money_market_rate_pct,
                risk_free_rate_pct, risk_free_rate_source]
    """
    df_raw = melt_imf_wide(path, value_col_name="rate_value")
    df_raw = df_raw[df_raw["country"].isin(TARGET_COUNTRIES)].copy()

    if "SCALE" in df_raw.columns:
        df_raw = df_raw.drop(columns=["SCALE"])

    # Assign candidate label based on indicator substring
    def _label(ind: str) -> str | None:
        if pd.isna(ind):
            return None
        for label, substr in RATE_CANDIDATES.items():
            if substr in ind:
                return label
        return None

    df_raw["rate_label"] = df_raw["indicator"].apply(_label)
    df_raw = df_raw.dropna(subset=["rate_label"])

    # Pivot: one column per candidate
    df_pivot = (
        df_raw
        .pivot_table(index=["country", "date"], columns="rate_label",
                     values="rate_value", aggfunc="first")
        .reset_index()
    )

    # Build composite risk-free rate (first non-NaN in priority order)
    df_pivot["risk_free_rate_pct"] = np.nan
    df_pivot["risk_free_rate_source"] = ""
    for col in RATE_PRIORITY:
        if col not in df_pivot.columns:
            continue
        mask = df_pivot["risk_free_rate_pct"].isna() & df_pivot[col].notna()
        df_pivot.loc[mask, "risk_free_rate_pct"] = df_pivot.loc[mask, col]
        df_pivot.loc[mask, "risk_free_rate_source"] = col

    return df_pivot

# 6. Load IMF Debt Data

### Variable candidates:

**Domestic-currency debt (Billions of domestic currency):**
1. **[CHOSEN] Gross debt, General government** — total govt debt stock
2. Net debt, General government — debt minus financial assets

**External debt (Billions of USD):**
3. **[CHOSEN] External debt, US dollar** — foreign-currency obligations

### Selection rationale
CCA requires separating domestic-currency liabilities (distress barrier
denominated in LCU) from foreign-currency liabilities (USD-denominated
distress barrier).  We use:
  - Gross debt as the total domestic liability proxy (conservative; net debt
    would understate obligations if financial assets are illiquid).
  - External debt in USD as the foreign-currency liability.

Note: Data is ANNUAL.  We forward-fill to weekly in the merge step.

In [54]:
DEBT_CANDIDATES = {
    # candidate_label                : indicator_substring
    "gross_debt_lcu_bn":              "Gross debt, General government, Domestic currency",
    "net_debt_lcu_bn":                "Net debt, General government, Domestic currency",
    "external_debt_usd_bn":           "External debt, US dollar",
}

# ── Chosen columns ──────────────────────────────────────────────────────────
# [CHOSEN] gross_debt_lcu_bn   — conservative total domestic debt
# [ALT]    net_debt_lcu_bn     — nets out govt financial assets
# [CHOSEN] external_debt_usd_bn — foreign-currency obligations


def load_debt_data(path: str) -> pd.DataFrame:
    """
    Load sovereign debt data (annual frequency).

    Returns
    -------
    DataFrame: [country, date, gross_debt_lcu_bn, net_debt_lcu_bn,
                external_debt_usd_bn]
    """
    df_raw = melt_imf_wide(path, value_col_name="debt_value")
    df_raw = df_raw[df_raw["country"].isin(TARGET_COUNTRIES)].copy()

    if "SCALE" in df_raw.columns:
        df_raw = df_raw.drop(columns=["SCALE"])

    # Assign candidate label
    def _label(ind: str) -> str | None:
        if pd.isna(ind):
            return None
        for label, substr in DEBT_CANDIDATES.items():
            if substr in ind:
                return label
        return None

    df_raw["debt_label"] = df_raw["indicator"].apply(_label)
    df_raw = df_raw.dropna(subset=["debt_label"])

    # Pivot
    df_pivot = (
        df_raw
        .pivot_table(index=["country", "date"], columns="debt_label",
                     values="debt_value", aggfunc="first")
        .reset_index()
    )

    return df_pivot

# 6b. Load QEDS External Debt by Maturity

Source: QEDS_SDDS.CSV (World Bank / IMF Quarterly External Debt Statistics)
Format: semicolon-separated, same wide layout as IMF files (timeseries per row),
        IMF-style country names.

### Variable candidates (all in USD, units TBD from data):
1. **[CHOSEN] Gross Ext. Debt Pos., General Government, Short-term, All instruments, USD**
   → Short-term external sovereign obligations (maturity < 1 year)
2. **[CHOSEN] Gross Ext. Debt Pos., General Government, Long-term, All instruments, USD**
   → Long-term external sovereign obligations (maturity ≥ 1 year)

### Selection rationale
CCA distress barriers distinguish between short-term debt (due within the
pricing horizon, enters the default barrier at face value) and long-term debt
(discounted; typically half enters the barrier following Moody's KMV convention).
Having the maturity split lets us calibrate the barrier more precisely than
using total external debt alone.

In [55]:
QEDS_CANDIDATES = {
    # column_label             : exact indicator name in QEDS
    "short_term_external_debt": "Gross Ext. Debt Pos., Publicly Guar. Private Sector Ext. Debt, Short-term, All instruments, USD",
    "long_term_external_debt":  "Gross Ext. Debt Pos., Publicly Guar. Private Sector Ext. Debt, Long-term, All instruments, USD",
}


def load_qeds_external_debt(path: str) -> pd.DataFrame:
    """
    Load QEDS external debt split by maturity.

    File format: semicolon-separated CSV with columns:
      - "Cleaned_Name"   : country name (already clean, similar to FX names)
      - "Indicator Name"  : full indicator description
      - Quarter columns  : "2003Q1", "2003Q2", ... (6-char, YYYYQn format)

    Candidate variables (USD)
    -------------------------
    [CHOSEN] "Gross Ext. Debt Pos., General Government, Short-term, All instruments, USD"
        → Short-term external government debt (maturity < 1 year).
        CCA usage: enters the default barrier at full face value.
    [CHOSEN] "Gross Ext. Debt Pos., General Government, Long-term, All instruments, USD"
        → Long-term external government debt (maturity ≥ 1 year).
        CCA usage: typically 50% enters the barrier (Moody's KMV convention).

    Returns
    -------
    DataFrame: [country, date, short_term_external_debt, long_term_external_debt]
    """
    df = pd.read_csv(path, delimiter=";")

    # ── Identify quarter columns (6-char strings containing 'Q') ────────────
    quarter_cols = [
        c for c in df.columns
        if c not in ["Cleaned_Name", "Indicator Name"]
        and len(c) == 6 and "Q" in c
    ]
    print(f"  QEDS: {len(quarter_cols)} quarters: {quarter_cols[0]} → {quarter_cols[-1]}")

    # ── Melt to long format ─────────────────────────────────────────────────
    df_long = df.melt(
        id_vars=["Cleaned_Name", "Indicator Name"],
        value_vars=quarter_cols,
        var_name="quarter",
        value_name="value",
    )
    df_long.columns = ["country", "indicator", "quarter", "value"]

    # ── Parse quarter strings to end-of-quarter dates ───────────────────────
    df_long["date"] = pd.to_datetime(
        df_long["quarter"]
        .str.replace("Q1", "-03-31", regex=False)
        .str.replace("Q2", "-06-30", regex=False)
        .str.replace("Q3", "-09-30", regex=False)
        .str.replace("Q4", "-12-31", regex=False),
        errors="coerce",
    )
    df_long["value"] = pd.to_numeric(df_long["value"], errors="coerce")
    df_long = df_long.dropna(subset=["date"])

    print(f"  QEDS long: {len(df_long):,} rows, {df_long['country'].nunique()} countries")

    # ── Match indicators (exact then case-insensitive fallback) ─────────────
    available = df_long["indicator"].unique()
    matched = {}
    for key, pattern in QEDS_CANDIDATES.items():
        if pattern in available:
            matched[key] = pattern
        else:
            # Case-insensitive fallback
            matches = [ind for ind in available if pattern.lower() in ind.lower()]
            if matches:
                matched[key] = matches[0]
                print(f"  ⚠ Partial match for {key}: '{matches[0]}'")

    print(f"  QEDS matched indicators: {list(matched.keys())}")
    if not matched:
        print("  ⚠ WARNING: No QEDS indicators matched — returning empty DataFrame")
        return pd.DataFrame(columns=["country", "date"])

    # ── Filter to matched indicators and pivot ──────────────────────────────
    inv_matched = {v: k for k, v in matched.items()}
    debt_data = df_long[df_long["indicator"].isin(matched.values())].copy()
    debt_data["var"] = debt_data["indicator"].map(inv_matched)

    # Standardise country names to FX convention
    debt_data["country"] = debt_data["country"].apply(standardize_imf_country)

    # Keep target countries
    debt_data = debt_data[debt_data["country"].isin(TARGET_COUNTRIES)].copy()

    df_pivot = (
        debt_data
        .pivot_table(
            index=["country", "date"],
            columns="var",
            values="value",
            aggfunc="first",
        )
        .reset_index()
    )

    print(f"  QEDS pivot: {df_pivot.shape[0]:,} rows, {df_pivot['country'].nunique()} countries")
    return df_pivot
#
# Strategy:
# 1. Start with the FX panel (weekly dates × countries) — this is the spine.
# 2. For each IMF dataset (monthly or annual), do an **as-of merge**
#    (merge_asof with forward direction) so that each weekly date picks up
#    the most recent available value.  This is equivalent to forward-filling
#    within each country.
# 3. Result: a single DataFrame indexed by (country, date) at weekly frequency.

In [56]:
def merge_panel(
    fx: pd.DataFrame,
    monetary_base: pd.DataFrame,
    rates: pd.DataFrame,
    debt: pd.DataFrame,
    qeds: pd.DataFrame,
) -> pd.DataFrame:
    """
    Merge all data sources into the weekly CCA panel.

    The FX DataFrame defines the (country, date) spine.
    Lower-frequency data is forward-filled via merge_asof per country.
    """
    # Ensure dates are datetime
    for df in [fx, monetary_base, rates, debt, qeds]:
        df["date"] = pd.to_datetime(df["date"])

    # Sort everything (required for merge_asof)
    fx = fx.sort_values(["country", "date"])
    monetary_base = monetary_base.sort_values(["country", "date"])
    rates = rates.sort_values(["country", "date"])
    debt = debt.sort_values(["country", "date"])
    qeds = qeds.sort_values(["country", "date"])

    panel = fx.copy()

    # ── Helper: per-country asof merge ──────────────────────────────────────
    def _asof_merge(left: pd.DataFrame, right: pd.DataFrame,
                    on_cols: list[str]) -> pd.DataFrame:
        """merge_asof by country, forward-filling from right into left."""
        merged_parts = []
        for country in left["country"].unique():
            l = left[left["country"] == country].copy()
            r = right[right["country"] == country].copy()
            if r.empty:
                # No data for this country — fill with NaN
                for c in on_cols:
                    l[c] = np.nan
                merged_parts.append(l)
                continue
            m = pd.merge_asof(
                l.sort_values("date"),
                r[["date"] + on_cols].sort_values("date"),
                on="date",
                direction="backward",   # pick most recent observation ≤ date
            )
            merged_parts.append(m)
        return pd.concat(merged_parts, ignore_index=True)

    # ── Merge monetary base (monthly → weekly) ──────────────────────────────
    mb_cols = [c for c in monetary_base.columns if c not in ("country", "date")]
    panel = _asof_merge(panel, monetary_base, mb_cols)

    # ── Merge risk-free rates (monthly → weekly) ────────────────────────────
    rate_cols = [c for c in rates.columns if c not in ("country", "date")]
    panel = _asof_merge(panel, rates, rate_cols)

    # ── Merge debt data (annual → weekly) ───────────────────────────────────
    debt_cols = [c for c in debt.columns if c not in ("country", "date")]
    panel = _asof_merge(panel, debt, debt_cols)

    # ── Merge QEDS external debt by maturity (quarterly → weekly) ───────────
    qeds_cols = [c for c in qeds.columns if c not in ("country", "date")]
    panel = _asof_merge(panel, qeds, qeds_cols)

    # ── Set double index ────────────────────────────────────────────────────
    panel = panel.sort_values(["country", "date"]).reset_index(drop=True)

    return panel

# 8. Derived CCA Variables

Compute additional columns needed for the CCA model.

In [57]:
def add_cca_derived_columns(panel: pd.DataFrame) -> pd.DataFrame:
    """
    Add derived columns useful for CCA estimation.

    Columns added
    -------------
    - gross_debt_lcu_mn : gross debt converted from billions to millions
                          (same unit as monetary base)
    - external_debt_lcu_mn : external debt converted from USD bn to LCU mn
                             using the FX rate
    - total_debt_lcu_mn : gross domestic + external (both in LCU millions)
    - risk_free_rate_decimal : risk-free rate as a decimal (e.g. 0.05 for 5%)
    """
    df = panel.copy()

    # ── Convert gross debt from billions to millions ────────────────────────
    if "gross_debt_lcu_bn" in df.columns:
        df["gross_debt_lcu_mn"] = df["gross_debt_lcu_bn"] * 1_000
    
    # ── Convert net debt from billions to millions ──────────────────────────
    if "net_debt_lcu_bn" in df.columns:
        df["net_debt_lcu_mn"] = df["net_debt_lcu_bn"] * 1_000

    # ── Convert external debt from USD bn to LCU mn ─────────────────────────
    #    external_debt_usd_bn × fx_rate_per_usd × 1000 = LCU millions
    if "external_debt_usd_bn" in df.columns and "fx_rate_per_usd" in df.columns:
        df["external_debt_lcu_mn"] = (
            df["external_debt_usd_bn"] * df["fx_rate_per_usd"] * 1_000
        )

    # ── Total debt in LCU millions ──────────────────────────────────────────
    #    [CHOSEN] gross_debt + external (converted)
    #    [ALT]    net_debt + external
    if "gross_debt_lcu_mn" in df.columns and "external_debt_lcu_mn" in df.columns:
        df["total_debt_lcu_mn"] = (
            df["gross_debt_lcu_mn"].fillna(0) + df["external_debt_lcu_mn"].fillna(0)
        )

    # ── Risk-free rate as decimal ───────────────────────────────────────────
    if "risk_free_rate_pct" in df.columns:
        df["risk_free_rate_decimal"] = df["risk_free_rate_pct"] / 100.0

    return df

# 9. Diagnostics & Summary

In [60]:
def print_panel_diagnostics(panel: pd.DataFrame) -> None:
    """Print coverage diagnostics for the CCA panel."""
    print("=" * 70)
    print("CCA PANEL DIAGNOSTICS")
    print("=" * 70)
    print(f"Shape: {panel.shape}")
    print(f"Countries: {panel['country'].nunique()}")
    print(f"Date range: {panel['date'].min().date()} → {panel['date'].max().date()}")
    print(f"Weeks: {panel['date'].nunique()}")
    print()

    # Per-column coverage
    print("Column coverage (% non-null):")
    for col in panel.columns:
        if col in ("country", "date"):
            continue
        pct = panel[col].notna().mean() * 100
        print(f"  {col:35s}  {pct:6.1f}%")
    print()

    # Per-country coverage of key variables
    key_vars = [v for v in panel.columns]

    print("Per-country coverage (% non-null) for key CCA variables:")
    coverage = (
        panel.groupby("country")[key_vars]
        .apply(lambda g: g.notna().mean() * 100)
    )
    print(coverage.round(1).to_string())
    print()

    # Countries with no data for key variables
    for var in key_vars:
        missing = coverage[coverage[var] == 0].index.tolist()
        if missing:
            print(f"  ⚠ Countries with NO data for {var}: {missing}")

# 10. Main Execution

In [ ]:
def main():
    print("Loading FX rates...")
    fx = load_fx_rates(os.path.join(DATA_DIR, "Weekly_FX_Rates.csv"))
    print(f"  → {len(fx):,} rows, {fx['country'].nunique()} countries")

    print("Loading CDS spreads...")
    cds = load_cds_spreads(os.path.join(DATA_DIR, "Weekly_CDS.csv"))
    print(f"  → {len(cds):,} rows, {cds['country'].nunique()} countries")

    print("Loading monetary base...")
    mb = load_monetary_base(os.path.join(DATA_DIR, "IMF_monetary_base_in_domestic_curr.csv"))
    print(f"  → {len(mb):,} rows, {mb['country'].nunique()} countries")

    print("Loading risk-free rates...")
    rates = load_risk_free_rates(os.path.join(DATA_DIR, "IMF_foreign_risk_free_rates.csv"))
    print(f"  → {len(rates):,} rows, {rates['country'].nunique()} countries")

    print("Loading debt data...")
    debt = load_debt_data(os.path.join(DATA_DIR, "IMF_debt_local_and_ext.csv"))
    print(f"  → {len(debt):,} rows, {debt['country'].nunique()} countries")

    print("Loading QEDS external debt by maturity...")
    qeds = load_qeds_external_debt(os.path.join(DATA_DIR, "QEDS_SDDS.CSV"))
    print(f"  → {len(qeds):,} rows, {qeds['country'].nunique()} countries")

    print("\nMerging into weekly panel...")
    panel = merge_panel(fx, mb, rates, debt, qeds)

    print("Adding derived CCA columns...")
    panel = add_cca_derived_columns(panel)

    # ── Save ────────────────────────────────────────────────────────────────
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    panel.to_csv(OUTPUT_FILE, index=False)
    print(f"\n✅ Panel saved to {OUTPUT_FILE}")
    print(f"   Shape: {panel.shape}")

    return panel


if __name__ == "__main__":
    panel = main()

Loading FX rates...
  → 254,660 rows, 70 countries
Loading monetary base...
  → 12,900 rows, 43 countries
Loading risk-free rates...
  → 12,711 rows, 48 countries
Loading debt data...
  → 1,489 rows, 60 countries
Loading QEDS external debt by maturity...
  QEDS: 110 quarters: 1998Q1 → 2025Q2
  QEDS long: 25,542,000 rows, 129 countries
  QEDS matched indicators: ['short_term_external_debt', 'long_term_external_debt']
  QEDS pivot: 1,917 rows, 54 countries
  → 1,917 rows, 54 countries

Merging into weekly panel...
Adding derived CCA columns...

✅ Panel saved to ../data/processed/CCA\cca_panel.csv
   Shape: (254660, 17)


In [64]:
# Filter countries and dates:

COUNTRIES = [ "Brazil", "Chile", "China", "Colombia"] #"Czechia", "Egypt",
                             #"Greece", "Hungary", "India", "Indonesia", "South Korea", "Kuwait", 
                             #"Malaysia", "Mexico", "Peru", "Philippines", "Poland", "Qatar",
                            #"Saudi Arabia", "South Africa", "Taiwan", "Thailand", "Turkey", "United Arab Emirates"]

START_DATE = '2014-01-01'
END_DATE = '2024-12-31'


panel_filtered = panel[panel['country'].isin(COUNTRIES)].copy()

# Filter by date range
panel_filtered = panel_filtered[
    (panel_filtered['date'] >= START_DATE) & 
    (panel_filtered['date'] <= END_DATE)
]


print_panel_diagnostics(panel_filtered)


CCA PANEL DIAGNOSTICS
Shape: (6888, 17)
Countries: 4
Date range: 2014-01-01 → 2024-12-30
Weeks: 1722

Column coverage (% non-null):
  fx_rate_per_usd                       100.0%
  monetary_base_lcu_mn                   75.0%
  govt_bond_short_pct                     0.0%
  money_market_rate_pct                  75.0%
  policy_rate_pct                        75.0%
  tbill_3m_pct                            0.0%
  risk_free_rate_pct                     75.0%
  risk_free_rate_source                  75.0%
  gross_debt_lcu_bn                      75.0%
  net_debt_lcu_bn                        75.0%
  long_term_external_debt                92.1%
  short_term_external_debt               92.1%
  gross_debt_lcu_mn                      75.0%
  net_debt_lcu_mn                        75.0%
  risk_free_rate_decimal                 75.0%

Per-country coverage (% non-null) for key CCA variables:
           date  country  fx_rate_per_usd  monetary_base_lcu_mn  govt_bond_short_pct  money_market_rate_p

In [46]:
panel = panel.dropna(subset=['fx_rate_per_usd', 'monetary_base_lcu_mn', 'risk_free_rate_pct', 'gross_debt_lcu_bn', 'short_term_external_debt', 'long_term_external_debt'], how='all')
panel

,date,country,fx_rate_per_usd,monetary_base_lcu_mn,govt_bond_short_pct,money_market_rate_pct,policy_rate_pct,tbill_3m_pct,risk_free_rate_pct,risk_free_rate_source,gross_debt_lcu_bn,net_debt_lcu_bn,long_term_external_debt,short_term_external_debt,gross_debt_lcu_mn,net_debt_lcu_mn,risk_free_rate_decimal


# Appendix: Column Reference

| Column | Source | Unit | Frequency | Notes |
|--------|--------|------|-----------|-------|
| `country` | — | — | — | FX-file naming convention |
| `date` | FX CSV | — | Weekly | Panel spine |
| `fx_rate_per_usd` | FX CSV | LCU/USD | Weekly | Direct quote |
| `monetary_base_lcu_mn` | IMF IFS | Millions LCU | Monthly→fwd-fill | CBS monetary base |
| `policy_rate_pct` | IMF IFS | % p.a. | Monthly→fwd-fill | Candidate |
| `tbill_3m_pct` | IMF IFS | % p.a. | Monthly→fwd-fill | Candidate (preferred) |
| `govt_bond_short_pct` | IMF IFS | % p.a. | Monthly→fwd-fill | Candidate |
| `money_market_rate_pct` | IMF IFS | % p.a. | Monthly→fwd-fill | Candidate |
| `risk_free_rate_pct` | Composite | % p.a. | Monthly→fwd-fill | Best available rate |
| `risk_free_rate_source` | — | — | — | Which candidate was used |
| `gross_debt_lcu_bn` | IMF WEO | Billions LCU | Annual→fwd-fill | [CHOSEN] |
| `net_debt_lcu_bn` | IMF WEO | Billions LCU | Annual→fwd-fill | [ALT] |
| `external_debt_usd_bn` | IMF WEO | Billions USD | Annual→fwd-fill | [CHOSEN] |
| `short_term_external_debt` | QEDS SDDS | USD | Quarterly→fwd-fill | [CHOSEN] Govt ext debt, maturity < 1yr |
| `long_term_external_debt` | QEDS SDDS | USD | Quarterly→fwd-fill | [CHOSEN] Govt ext debt, maturity ≥ 1yr |
| `gross_debt_lcu_mn` | Derived | Millions LCU | Weekly | = gross_debt_lcu_bn × 1000 |
| `net_debt_lcu_mn` | Derived | Millions LCU | Weekly | = net_debt_lcu_bn × 1000 |
| `external_debt_lcu_mn` | Derived | Millions LCU | Weekly | = ext_debt_usd_bn × fx × 1000 |
| `total_debt_lcu_mn` | Derived | Millions LCU | Weekly | gross_dom + ext (in LCU) |
| `risk_free_rate_decimal` | Derived | decimal | Weekly | = rate_pct / 100 |

In [41]:
panel[panel['country']=='Brazil']

,date,country,fx_rate_per_usd,monetary_base_lcu_mn,govt_bond_short_pct,money_market_rate_pct,policy_rate_pct,tbill_3m_pct,risk_free_rate_pct,risk_free_rate_source,gross_debt_lcu_bn,net_debt_lcu_bn,long_term_external_debt,short_term_external_debt,gross_debt_lcu_mn,net_debt_lcu_mn,risk_free_rate_decimal
21828,2000-01-01,Brazil,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
21829,2000-01-08,Brazil,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
21830,2000-01-15,Brazil,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
21831,2000-01-22,Brazil,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
21832,2000-01-29,Brazil,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25461,2024-12-21,Brazil,6.0842,1312168.43,NaN,11.04,11.25,NaN,11.25,policy_rate_pct,9192.466,6612.83,954538385.7,129214335.6,9192466.0,6612830.0,0.1125
25462,2024-12-23,Brazil,6.1937,1312168.43,NaN,11.04,11.25,NaN,11.25,policy_rate_pct,9192.466,6612.83,954538385.7,129214335.6,9192466.0,6612830.0,0.1125
25463,2024-12-25,Brazil,6.1937,1312168.43,NaN,11.04,11.25,NaN,11.25,policy_rate_pct,9192.466,6612.83,954538385.7,129214335.6,9192466.0,6612830.0,0.1125
25464,2024-12-28,Brazil,6.1937,1312168.43,NaN,11.04,11.25,NaN,11.25,policy_rate_pct,9192.466,6612.83,954538385.7,129214335.6,9192466.0,6612830.0,0.1125
